# Notebook File Overview
This notebook features two different SQL queries:

**The first SQL query extracts variables we chose from Conditions domain**

**The second SQL query extracts variables we chose from Visits domain**

Each query is autogenerated from the details we provided during the cohort builder and dataset builder phase which creates the tables based on the Concept Sets we selected:
1.	Conditions
2.	Visits

# SQL Code Chunk to Extract Condition Variables

This SQL query extracts the condirtion variables we selected in the dataset builder concept sets:

* diabetes diagnosis dates
* Chronic Kidney Disease (CKD) status
* Chronic Kidney Disease (CKD) diagnosis


In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "diabetes_ckd_surv" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_47085065_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (201820, 37017104, 37017813, 37018761, 43531559, 43531562, 43531566, 43531577, 43531578, 43531653, 443597, 443601, 443611, 443612, 443614, 44782429, 44782689, 44782690, 44782691, 44782692, 45763854, 45763855, 45769901, 45769903, 45771075, 45773576, 45773688, 46270353, 46271022, 46273636)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_47085065_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_47085065",
  "condition_47085065_*.csv")
message(str_glue('The data will be written to {condition_47085065_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_47085065_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_47085065_path,
  destination_format = "CSV")



In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_47085065_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_47085065_condition_df <- read_bq_export_from_workspace_bucket(condition_47085065_path)

dim(dataset_47085065_condition_df)

head(dataset_47085065_condition_df, 5)

# SQL Code Chunk to Extract Visit Variables

This SQL query extracts the Visit variables we selected in the dataset builder concept sets:

* This give us each visit date by person_id and viisit type
* We will use this data to create the date value for people in our cohort who did not have CKD so we can see when they were last follow-ed up with

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "diabetes_ckd_surv" for domain "visit" and was generated for All of Us Controlled Tier Dataset v8
dataset_47085065_visit_sql <- paste("
    SELECT
        visit.PERSON_ID,
        v_standard_concept.concept_name as standard_concept_name,
        visit.visit_start_datetime,
        visit.visit_end_datetime 
    FROM
        `visit_occurrence` visit 
    LEFT JOIN
        `concept` v_standard_concept 
            ON visit.visit_concept_id = v_standard_concept.concept_id 
    WHERE
        (
            visit_concept_id IN (5083, 581477, 9201, 9202, 9203)
        )  
        AND (
            visit.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
visit_47085065_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "visit_47085065",
  "visit_47085065_*.csv")
message(str_glue('The data will be written to {visit_47085065_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_47085065_visit_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  visit_47085065_path,
  destination_format = "CSV")



In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {visit_47085065_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_47085065_visit_df <- read_bq_export_from_workspace_bucket(visit_47085065_path)

dim(dataset_47085065_visit_df)

head(dataset_47085065_visit_df, 5)

# Arrange the datasets

in these code chunks we will see the structure of our datasets and arrange them by person_id to get an idea of how many visits a person has had

In [ ]:
#Arrange condiiton data by person_id
condition_arranged <- dataset_47085065_condition_df %>%
arrange(person_id)

dim(condition_arranged)
head(condition_arranged, 10)


#Arrange visit data by person_id
visit_arranged <- dataset_47085065_visit_df %>%
arrange(PERSON_ID) %>%
select(PERSON_ID, visit_end_datetime)

dim(visit_arranged)
head(visit_arranged, 10)

# Run the provided snippets to save our 2 arranged datasets to our workspace bucket

This section saves R data frames that are already in memory to our workspace bucket using a two-step process:

1. Writes dataframe to local CSV file on persistent disk
2. Copies file from local storage to workspace bucket using gsutil

It also verifies successful upload by listing bucket contents

Once these datasets are saved to the workspace bucket, we no longer need our persistent disk

**Please note that we will be updating this code ourselves and even chaging the code after the warning**

<div style=" background-color:#f8d7da; color:#842029; border:1px solid #f5c2c7; border-radius:8px; padding:20px; font-size:20px; "> 
<b>We are going to update the provided snippet, even past the <i>DON'T CHANGE FROM HERE</i> Warning</b><br> 
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
condition_arranged <- condition_arranged
visit_arranged <- visit_arranged

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
condition_arranged_filename <- 'condition_arranged.csv'
visit_arranged_filename <- 'visit_arranged.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(condition_arranged, condition_arranged_filename)
write_excel_csv(visit_arranged, visit_arranged_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", condition_arranged_filename, " ", my_bucket, "/data/"), intern=T)
system(paste0("gsutil cp ./", visit_arranged_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)

<div class="alert alert-block alert-danger" style="font-size:24px; padding: 10px;">
    <b>Once you save to the workspace bucket, you do not need to run the queries again</b>
    <ul style="font-size:20px; margin-top: 5px;">
        <li>Recommend using a new Jupyter Notebook to read in saved data for analysis</li>
    </ul>
</div>